## **Comparing Random and Real Network Models**

In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import collections

## 1. Building the Networks

In [ ]:
n = 1000
p = 0.01
G_random = nx.erdos_renyi_graph(n, p, seed=42)
G_ba = nx.barabasi_albert_graph(n, 3, seed=42)
G_real = nx.karate_club_graph()
print(f" {G_random.number_of_nodes()}, Edges: {G_random.number_of_edges()}")
print(f" {G_ba.number_of_nodes()}, Edges: {G_ba.number_of_edges()}")
print(f" {G_real.number_of_nodes()}, Edges: {G_real.number_of_edges()}")

## 2. Basic Statistics Comparison

In [ ]:
def get_stats(G, name):
    avg_degree = sum(dict(G.degree()).values()) / G.number_of_nodes()
    clustering = nx.average_clustering(G)
    if nx.is_connected(G):
        avg_path = nx.average_shortest_path_length(G)
    else:
        largest_cc = G.subgraph(max(nx.connected_components(G), key=len))
        avg_path = nx.average_shortest_path_length(largest_cc)
    print(f"{name}")
    print(f"  Avg {avg_degree:.4f}")
    print(f"  Clustering Coeff {clustering:.4f}")
    print(f"  Avg Path Length {avg_path:.4f}")
    print()
get_stats(G_random, "Erdos-Renyi Random")
get_stats(G_ba, "Barabasi-Albert")
get_stats(G_real, "Karate Club (Real)")

## 3. Degree Distribution

In [ ]:
def plot_degree_dist(G, ax, title, color):
    degree_sequence = sorted([d for n, d in G.degree()], reverse=True)
    degree_count = collections.Counter(degree_sequence)
    deg, cnt = zip(*sorted(degree_count.items()))
    
    ax.bar(deg, cnt, color=color, alpha=0.7, edgecolor='black', linewidth=0.4)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Degree")
    ax.set_ylabel("Count")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

plot_degree_dist(G_random, axes[0], "Erdos-Renyi Random", "steelblue")
plot_degree_dist(G_ba, axes[1], "Barabasi-Albert", "darkorange")
plot_degree_dist(G_real, axes[2], "Karate Club (Real)", "seagreen")

plt.suptitle("Degree Distribution Comparison", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("degree_distribution.png", dpi=150)
plt.show()

## 4. Log-Log Degree Distribution (Power Law Check)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, G, title, color in zip(
    axes,
    [G_random, G_ba, G_real],
    ["Erdos-Renyi Random", "Barabasi-Albert", "Karate Club (Real)"],
    ["steelblue", "darkorange", "seagreen"]
):
    degree_sequence = sorted([d for n, d in G.degree()], reverse=True)
    degree_count = collections.Counter(degree_sequence)
    deg, cnt = zip(*sorted(degree_count.items()))
    
    ax.loglog(deg, cnt, 'o', color=color, markersize=5, alpha=0.8)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Degree (log)")
    ax.set_ylabel("Count (log)")
    ax.grid(True, which='both', linestyle='--', linewidth=0.4, alpha=0.5)

plt.suptitle("Log-Log Degree Distribution", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("loglog_degree.png", dpi=150)
plt.show()

## 5. Network Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
G_random_sub = G_random.subgraph(list(G_random.nodes())[:100])
G_ba_sub = G_ba.subgraph(list(G_ba.nodes())[:100])
for ax, G, title in zip(
    axes,
    [G_random_sub, G_ba_sub, G_real],
    ["Erdos-Renyi (sample 100)", "Barabasi-Albert (sample 100)", "Karate Club (Real)"]
):
    pos = nx.spring_layout(G, seed=42)
    degrees = dict(G.degree())
    node_sizes = [20 + degrees[n] * 10 for n in G.nodes()]
    nx.draw_networkx(
        G, pos, ax=ax,
        with_labels=False,
        node_size=node_sizes,
        node_color='steelblue',
        edge_color='gray',
        alpha=0.7,
        width=0.5
    )
    ax.set_title(title, fontsize=11)
    ax.axis('off')
plt.suptitle("Network Structure Visualization", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("network_visualization.png", dpi=150)
plt.show()

## 6. Summary Table

In [ ]:
import pandas as pd

def summary(G):
    degrees = [d for n, d in G.degree()]
    clustering = nx.average_clustering(G)
    if nx.is_connected(G):
        avg_path = round(nx.average_shortest_path_length(G), 4)
    else:
        lcc = G.subgraph(max(nx.connected_components(G), key=len))
        avg_path = round(nx.average_shortest_path_length(lcc), 4)
    return {
        "Nodes": G.number_of_nodes(),
        "Edges": G.number_of_edges(),
        "Avg Degree": round(np.mean(degrees), 4),
        "Max Degree": max(degrees),
        "Clustering Coeff": round(clustering, 4),
        "Avg Path Length": avg_path,
        "Density": round(nx.density(G), 6)
    }

df = pd.DataFrame({
    "Erdos-Renyi": summary(G_random),
    "Barabasi-Albert": summary(G_ba),
    "Karate Club (Real)": summary(G_real)
}).T

print(df.to_string())

## Observations

- The Erdos-Renyi model produces a Poisson degree distribution - most nodes have similar degrees, no hubs.
- The Barabasi-Albert model shows a power-law distribution - few nodes with very high degree (hubs), which is typical of real-world networks.
- The real Karate Club network also shows heterogeneous degree distribution consistent with real social networks.
- Clustering coefficient is notably higher in real networks compared to purely random graphs.
- The small-world property (short average path length) appears in all three but is more pronounced in scale-free models.